In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report

)

In [ ]:
X_train = pd.read_csv("../data/processed/X_train_smote.csv")
y_train = pd.read_csv("../data/processed/y_train_smote.csv").squeeze()

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("Training Shape :", X_train.shape)
print("Testing Shape :", X_test.shape)

In [ ]:
models = {

    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        eval_metric="logloss"
    )

}

In [ ]:
results = []

trained_models = {}

def train_model(model, model_name):

    print("="*60)
    print(model_name)
    print("="*60)

    # Train
    model.fit(X_train, y_train)

    # Prediction
    y_pred = model.predict(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])  # probabilities, not 0/1 labels

    # Save metrics
    results.append({

        "Model": model_name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1,

        "ROC AUC": roc

    })

    trained_models[model_name] = model

    print(classification_report(y_test, y_pred))

    print("Confusion Matrix")

    print(confusion_matrix(y_test, y_pred))

In [ ]:
for name, model in models.items():

    train_model(model, name)

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
)

results_df

In [ ]:
results_df.to_csv("../reports/model_comparison.csv", index=False)

In [ ]:
for name, model in trained_models.items():

    filename = name.lower().replace(" ", "_") + ".pkl"

    joblib.dump(
        model,
        f"../models/{filename}"
    )

print("All models saved successfully.")

In [ ]:
best_name = results_df.iloc[0]["Model"]   # best F1 from the table above
best_model = trained_models[best_name]
print("Best model:", best_name)

joblib.dump(
    best_model,
    "../models/best_model.pkl"
)

print("Best model saved.")

In [ ]:
# Load model
model = joblib.load("../models/best_model.pkl")

# Load test data
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, y_pred):.6f}")
print(f"Precision: {precision_score(y_test, y_pred):.6f}")
print(f"Recall   : {recall_score(y_test, y_pred):.6f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.6f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_prob):.6f}")